In [123]:
import pandas as pd
import boto3, dotenv, os
import json, io

dotenv.load_dotenv(override=True)

s3 = boto3.client('s3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION')
)

bucket = os.getenv('AWS_BUCKET')

# --- Fetch the api json file ---
api_data = s3.get_object(Bucket=bucket, Key='api/data.json')
api_data:dict = json.loads(api_data['Body'].read().decode('utf-8'))

# --- Fetch the csv file ---
csv_data = s3.get_object(Bucket=bucket, Key='csv/pr.data.0.Current.csv')
csv_data = csv_data['Body'].read()


In [124]:
df_csv = pd.read_csv(io.BytesIO(csv_data),delimiter='\t')

df_api = pd.DataFrame(api_data['data'])
filter_api = (df_api['Year'] >= 2013) & (df_api['Year'] <= 2018)
df_api_filtered = df_api[filter_api]

print(f"Mean of the US annual population: {df_api_filtered['Population'].mean()}")
print(f"Standard deviation of the US annual population: {df_api_filtered['Population'].std()}")



Mean of the US annual population: 322069808.0
Standard deviation of the US annual population: 4158441.040908095


In [125]:
# --- Removing whitespace from column names ---
df_csv.rename(
    {
        'series_id        ': 'series_id',
        '       value': 'value'
    },
    axis = 1,
    inplace = True
)

for col in df_csv.columns:
    if df_csv[col].dtype == object:
        df_csv[col] = df_csv[col].str.strip()

In [126]:
print(f"Found {len(df_csv[df_csv['series_id'].isna()])} null values in series_id.")
print(f"Found {len(df_csv[df_csv['value'].isna()])} null values in value.")
print(f"Found {len(df_csv[df_csv['year'].isna()])} null values in year.")
print(f"Found {len(df_csv[df_csv['period'].isna()])} null values in period.")

Found 0 null values in series_id.
Found 0 null values in value.
Found 0 null values in year.
Found 0 null values in period.


In [127]:
# --- Aggregating the data by series_id and year for calculating the sum of the values ---
df_csv_agg = df_csv.groupby(['series_id', 'year']) \
    .agg({'value': 'mean'}) \
    .reset_index()

# --- Calculating the dense rank of the values ---
df_csv_agg['dense_rank'] = df_csv_agg.groupby(['series_id'])['value'] \
    .rank(
        ascending=False,
        method='dense'
    )

# --- Selecting the top 1 value for each series_id ---
cols = ['series_id', 'year', 'value']
df_csv_agg[df_csv_agg['dense_rank'] == 1][cols].reset_index(drop=True)

,series_id,year,value
0,PRS30006011,2022,4.1000
1,PRS30006012,2022,3.4200
2,PRS30006013,1998,141.1790
3,PRS30006021,2010,3.5400
4,PRS30006022,2010,2.4800
...,...,...,...
281,PRS88003192,2002,56.5600
282,PRS88003193,2025,172.7264
283,PRS88003201,2022,7.7800
284,PRS88003202,2022,5.9400


In [128]:
df_filtered = df_csv[(df_csv['series_id'] == 'PRS30006032') & (df_csv['period'] == 'Q01')]

df_merged = pd.merge(
    df_filtered,
    df_api,
    left_on = 'year',
    right_on = 'Year',
    how = 'inner'
)

cols_to_keep = ['series_id', 'year', 'period', 'value', 'Population']


df_merged[cols_to_keep]

,series_id,year,period,value,Population
0,PRS30006032,2013,Q01,0.5,316128839.0
1,PRS30006032,2014,Q01,-0.1,318857056.0
2,PRS30006032,2015,Q01,-1.7,321418821.0
3,PRS30006032,2016,Q01,-1.4,323127515.0
4,PRS30006032,2017,Q01,0.9,325719178.0
5,PRS30006032,2018,Q01,0.5,327167439.0
6,PRS30006032,2019,Q01,-1.6,328239523.0
7,PRS30006032,2021,Q01,0.5,331893745.0
8,PRS30006032,2022,Q01,5.3,333287562.0
9,PRS30006032,2023,Q01,0.5,334914896.0
